# Housing Data Analysis and Price Prediction

## Project Overview

This notebook explores a housing dataset and builds a baseline linear regression model to predict the target column `MEDV`.

The workflow includes data inspection, descriptive statistics, visual exploration, correlation analysis, a train-test split, model training, and regression evaluation.

## Tools
- Python
- pandas and NumPy
- Matplotlib and Seaborn
- scikit-learn

## Before Running
Install the required packages with `pip install pandas numpy matplotlib seaborn scikit-learn`. Place `Housing.csv` in the same folder as this notebook, then run the cells from top to bottom.

The notebook expects the target column to be named `MEDV` and uses the remaining columns as model features.

## 1. Import libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load the dataset

In [ ]:
DATA_PATH = Path("Housing.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Place Housing.csv next to this notebook."
    )

housing = pd.read_csv(DATA_PATH)
housing.head()

## 3. Inspect the data

In [ ]:
# Review the dataset dimensions and column types.
print(f"Rows: {housing.shape[0]:,}")
print(f"Columns: {housing.shape[1]:,}")
housing.dtypes

In [ ]:
# Count missing values in each column.
housing.isna().sum()

In [ ]:
# Summary statistics for numeric columns.
housing.describe()

## 4. Explore the target variable

In [ ]:
housing["MEDV"].plot(
    kind="hist",
    figsize=(8, 5),
    title="Distribution of Housing Values",
    xlabel="Housing Value (MEDV)",
    ylabel="Frequency",
)
plt.tight_layout()
plt.show()

In [ ]:
housing["MEDV"].plot(
    kind="box",
    figsize=(8, 4),
    title="Housing Value Distribution and Potential Outliers",
    ylabel="MEDV",
)
plt.tight_layout()
plt.show()

## 5. Explore relationships and correlations

In [ ]:
correlation = housing.corr(numeric_only=True)

medv_correlation = (
    correlation["MEDV"]
    .drop("MEDV")
    .sort_values(key=lambda values: values.abs(), ascending=False)
)
medv_correlation

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between Numeric Housing Variables")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(housing["RM"], housing["MEDV"], alpha=0.6)
plt.title("Number of Rooms vs. Housing Value")
plt.xlabel("Average Number of Rooms (RM)")
plt.ylabel("Housing Value (MEDV)")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=housing, x="LSTAT", y="MEDV")
plt.title("LSTAT vs. Housing Value")
plt.xlabel("LSTAT")
plt.ylabel("Housing Value (MEDV)")
plt.tight_layout()
plt.show()

## 6. Compare housing values by CHAS group

In [ ]:
housing["CHAS"].value_counts(dropna=False)

In [ ]:
chas_comparison = housing.groupby("CHAS")["MEDV"].mean()
chas_comparison

In [ ]:
chas_comparison.plot(kind="bar", figsize=(8, 5))
plt.title("Average Housing Value by CHAS Group")
plt.xlabel("CHAS Group")
plt.ylabel("Average Housing Value (MEDV)")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

## 7. Prepare features and target

In [ ]:
# MEDV is the prediction target; all remaining columns are features.
X = housing.drop(columns=["MEDV"])
y = housing["MEDV"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

## 8. Split into training and test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
print("Training target:", y_train.shape)
print("Test target:", y_test.shape)

## 9. Train a linear regression model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

## 10. Generate predictions

In [ ]:
y_pred = model.predict(X_test)

comparison = pd.DataFrame({
    "Actual MEDV": y_test.to_numpy(),
    "Predicted MEDV": y_pred,
})
comparison.head()

## 11. Evaluate model performance

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

metrics = pd.Series({
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R²": r2,
}, name="Value")
metrics

## 12. Compare actual and predicted values

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred, alpha=0.7)

line_min = min(y_test.min(), y_pred.min())
line_max = max(y_test.max(), y_pred.max())
plt.plot(
    [line_min, line_max],
    [line_min, line_max],
    "r--",
    label="Perfect Prediction",
)

plt.title("Actual vs. Predicted Housing Values")
plt.xlabel("Actual MEDV")
plt.ylabel("Predicted MEDV")
plt.legend()
plt.tight_layout()
plt.show()

## Notes and Limitations

- This is a baseline linear regression model; results depend on the dataset and its features.
- The notebook reports evaluation metrics on a held-out test split. These metrics do not by themselves establish that the model generalizes to every housing market.
- The analysis uses the dataset as supplied and does not infer causal relationships from correlations.
- Ensure `Housing.csv` is available locally before running the notebook.